# Insertion from dataset to milvus/parquet

### Imports

In [ ]:
import numpy as np
import pandas as pd

from tqdm import tqdm

from pymilvus import MilvusClient, DataType

import scipy.sparse as sp
from rdkit.Chem import rdFingerprintGenerator
from skfp.fingerprints import MAPFingerprint, AtomPairFingerprint

from chemap import compute_fingerprints, FingerprintConfig

from config import Config

### Load configuration

In [ ]:
CONFIG = Config(
    DB_NAME="skfp_atompair_fp4096_folded_dense_count",
    RESET_DATA=True,
    RESET_DATABASE=True,
    BINARY_EMBEDDING=False,
)

### Load Dataset

In [ ]:
loaded_compounds = pd.read_csv(CONFIG.DATA_DIR_PATH / "120_subclasses_chemical_subset.csv")
smiles = loaded_compounds["SMILES"].tolist()
metadata = pd.Series(loaded_compounds.to_dict(orient="records"), dtype="object")

print(f"Total compounds loaded: {len(loaded_compounds)}")

### Define Insertion to Parquet 

In [ ]:
def insert_to_parquet(
    embeddings: list, experiment_name: str, data_path=CONFIG.DATA_DIR_PATH
):
    output_dir = data_path / experiment_name
    if not output_dir.exists():
        output_dir.mkdir(parents=True)
    df = pd.DataFrame(
        {
            "inchikey": loaded_compounds["inchikey"],
            "embedding": embeddings,
            "metadata": metadata,
        }
    )
    df.to_parquet(output_dir / "data.parquet", index=False)


# Chemap Embeddings

In [ ]:
# ----------------------------
# RDKit: Morgan (folded, dense, binary)
# ----------------------------
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
X_morgan = compute_fingerprints(
    smiles,
    morgan,
    config=FingerprintConfig(
        count=True,
        folded=True,
        return_csr=False,   # dense numpy
        invalid_policy="raise",
    ),
)

insert_to_parquet(list(X_morgan), "morgan_r3_fp4096_folded_dense_count")

In [ ]:
# ----------------------------
# RDKit: Morgan (folded, dense, count)
# ----------------------------
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
X_morgan = compute_fingerprints(
    smiles,
    morgan,
    config=FingerprintConfig(
        count=True,
        folded=True,
        return_csr=False,   # dense numpy
        invalid_policy="raise",
    ),
)

insert_to_parquet(list(X_morgan), "morgan_r3_fp4096_folded_dense_count")

In [ ]:
# -----------------------------------
# RDKit: RDKitFP (folded, CSR sparse)
# -----------------------------------
rdkitfp = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=4096)
X_rdkitfp_csr = compute_fingerprints(
    smiles,
    rdkitfp,
    config=FingerprintConfig(
        count=False,
        folded=True,
        return_csr=True,  # SciPy CSR
        invalid_policy="raise",
    ),
)
embeddings = [
    row.toarray().ravel().astype(np.float32).tolist()
    for row in X_rdkitfp_csr
    if type(row) is sp.csr_matrix
]
assert len(embeddings) == len(loaded_compounds), "Mismatch in number of embeddings and compounds."

insert_to_parquet(embeddings, "rdkitfp_r3_fp4096_folded_csr_sparse_binary")


In [ ]:
# --------------------------------------------------
# scikit-fingerprints: MAPFingerprint (folded, dense, count)
# --------------------------------------------------
# MAPFingerprint is a MinHash-like fingerprint (different from MAP4 lib).
map_fp = MAPFingerprint(radius=2, fp_size=4096, variant="count", sparse=False)
X_map = compute_fingerprints(
    smiles,
    map_fp,
    config=FingerprintConfig(
        count=True,
        folded=True,
        return_csr=False,
        invalid_policy="raise",
    ),
)

insert_to_parquet(list(X_map), "skfp_map_r2_fp4096_folded_dense_count")

In [ ]:
# ----------------------------------------------------
# scikit-fingerprints: AtomPairFingerprint (folded, dense, count)
# ----------------------------------------------------
atom_pair = AtomPairFingerprint(fp_size=4096, count=True, sparse=False, use_3D=False)
X_ap_csr = compute_fingerprints(
    smiles,
    atom_pair,
    config=FingerprintConfig(
        count=True,
        folded=True,
        return_csr=False,
        invalid_policy="raise",
    ),
)

insert_to_parquet(list(X_ap_csr), "skfp_atompair_fp4096_folded_dense_count")

# Insert to Milvus Database

**Make sure to run the milvus server file using ```docker compose up``` inside the ```milvus/``` directory**

Milvus Client

In [ ]:
try:
    client = MilvusClient(
        uri=CONFIG.MILVUS_URI,
        token=CONFIG.MILVUS_TOKEN,
        timeout=CONFIG.MILVUS_TIMEOUT,
    )
    version_info = client.get_server_version()
    assert version_info is not None, "Failed to retrieve server version"
    print(f"Milvus client connected. Server Version: {version_info}")
except Exception as e:
    print(f"Failed to connect to Milvus server: {str(e)}")
    raise e

Create or reset database

In [ ]:
try:
    if CONFIG.RESET_DATABASE and CONFIG.DB_NAME in client.list_databases():
        client.use_database(db_name=CONFIG.DB_NAME)
        collections = client.list_collections()
        for collection in collections:  # type: ignore
            client.drop_collection(collection_name=collection)
        client.drop_database(db_name=CONFIG.DB_NAME)
        print(f"Database \"{CONFIG.DB_NAME}\" reset successfully.")
    
    if CONFIG.DB_NAME not in client.list_databases():
        client.create_database(
            db_name=CONFIG.DB_NAME,
            description=CONFIG.DATABASE_DESCRIPTION,
        )

    db_list = client.list_databases()
    assert CONFIG.DB_NAME in db_list, f"Database \"{CONFIG.DB_NAME}\" was not created successfully."
    client.use_database(db_name=CONFIG.DB_NAME)
    print(f"Available databases: {db_list}")
except Exception as e:
    print(f"Database operation failed: {str(e)}")
    raise e

Create collection with scheme

In [ ]:
client.use_database(db_name=CONFIG.DB_NAME)
schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

schema.add_field(
    field_name="inchikey",
    datatype=DataType.VARCHAR,
    max_length=27,
    is_primary=True,
    auto_id=False,
)
schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=4096,
)
schema.add_field(
    field_name="metadata",
    datatype=DataType.JSON,
    max_length=65535,
)

index_params = client.prepare_index_params()

if CONFIG.BINARY_EMBEDDING:
    index_params.add_index(
        field_name="embedding", index_type="BIN_FLAT", metric_type="JACCARD"
    )
else:
    index_params.add_index(field_name="embedding", index_type="FLAT", metric_type="L2")

try:
    if CONFIG.RESET_DATA:
        if CONFIG.COLLECTION_NAME in client.list_collections():  # type: ignore
            client.drop_collection(collection_name=CONFIG.COLLECTION_NAME)
            print(f'Collection "{CONFIG.COLLECTION_NAME}" dropped successfully.')
        else:
            print(
                f'Collection "{CONFIG.COLLECTION_NAME}" does not exist. No need to drop.'
            )

        client.create_collection(
            collection_name=CONFIG.COLLECTION_NAME,
            schema=schema,
            index_params=index_params,
        )

    assert client.has_collection(collection_name=CONFIG.COLLECTION_NAME), (
        f'Collection "{CONFIG.COLLECTION_NAME}" was not found.'
    )
    assert (
        client.get_collection_stats(collection_name=CONFIG.COLLECTION_NAME) is not None
    ), f'Failed to retrieve stats for collection "{CONFIG.COLLECTION_NAME}".'
    print(f'Collection "{CONFIG.COLLECTION_NAME}" created successfully.')
except Exception as e:
    print(f"Collection operation failed: {str(e)}")


Define insertion pipeline

In [ ]:
def ingest_dataframe_with_milvus_client(
    df: pd.DataFrame,
    batch_size: int = 500,
    binary_vector: bool = CONFIG.BINARY_EMBEDDING,
):
    inchikeys = df["inchikey"].to_numpy()
    smiles_lst = df["SMILES"].to_numpy()
    metadata_lst = df.to_dict(orient="records")
    assert len(inchikeys) == len(smiles_lst), (
        "InChIKeys and SMILES lists must be of the same length."
    )

    client.use_database(db_name=CONFIG.DB_NAME)
    for i in tqdm(
        range(0, len(df), batch_size),
        unit="batch",
        desc="Ingesting data",
    ):
        fingerprints = df["embedding"].iloc[i : min(i + batch_size, len(df))].tolist()
        if binary_vector:
            fingerprints_array = np.array(fingerprints).astype(np.uint8)
            binary_vectors = np.packbits(fingerprints_array, axis=1)
            batch_inchikeys = inchikeys[i : min(i + batch_size, len(df))]
            batch_metadata = metadata_lst[i : min(i + batch_size, len(df))]
            batch = [
                {
                    "inchikey": batch_inchikeys[j],
                    "embedding": binary_vectors[j].tobytes(),
                    "metadata": batch_metadata[j],
                }
                for j in range(len(batch_inchikeys))
            ]
        else:
            fingerprints_array = np.array(fingerprints).astype(np.uint64)
            batch_inchikeys = inchikeys[i : min(i + batch_size, len(df))]
            batch_metadata = metadata_lst[i : min(i + batch_size, len(df))]
            batch = [
                {
                    "inchikey": batch_inchikeys[j],
                    "embedding": fingerprints_array[j].tolist(),
                    "metadata": batch_metadata[j],
                }
                for j in range(len(batch_inchikeys))
            ]

        client.insert(
            collection_name=CONFIG.COLLECTION_NAME,
            data=batch,
        )

    client.flush(collection_name=CONFIG.COLLECTION_NAME)


Trigger Insertion pipe with loaded data

In [ ]:
if CONFIG.RESET_DATA:
    # Change configuration to the desired fingerprint type for insertion
    loaded_compounds["embedding"] = list(
        compute_fingerprints(
            smiles,
            AtomPairFingerprint(fp_size=4096, count=True, sparse=False, use_3D=False),
            config=FingerprintConfig(
                count=True,
                folded=True,
                return_csr=False,
                invalid_policy="raise",
            ),
        )
    )
    ingest_dataframe_with_milvus_client(
        df=loaded_compounds, batch_size=1_000, binary_vector=CONFIG.BINARY_EMBEDDING
    )
